----------------------------------------------------------------------------------------------------

# ***Testando APIs RESTful Assíncronas em FastAPI***

----------------------------------------------------------------------------------------------------

# ***1. Reestruturando o projeto para realização de testes*** 

## ***1.1. Criação do arquivo 'conftest.py' na pasta 'tests'***

In [ ]:
import asyncio
import os

import pytest_asyncio
from httpx import ASGITransport, AsyncClient

os.environ.setdefault("DATABASE_URL", f"sqlite:///tests.db")


@pytest_asyncio.fixture
async def db(request):
    from src.app.database import database, engine, metadata
    from src.models.post import posts

    await database.connect()
    metadata.create_all(engine)

    def teardown():
        async def _teardown():
            await database.disconnect()
            metadata.drop_all(engine)

        asyncio.run(_teardown())

    request.addfinalizer(teardown)


## ***1.2. Atualização do arquivo 'database.py' na pasta 'src'***

In [ ]:
import os

import databases 
import sqlalchemy 

DATABASE_URL = os.getenv("DATABASE_URL",'sqlite:///./blog.db')

# cria database
database = databases.Database(DATABASE_URL)
# instanciando a classe SQLalchemmy
metadata = sqlalchemy.MetaData()
engine = sqlalchemy.create_engine(DATABASE_URL, connect_args = {"check_same_thread": False})

## ***1.3. Atualização do arquivo 'conftest.py' na pasta 'tests'***

In [ ]:
import asyncio
import os

import pytest_asyncio
from httpx import ASGITransport, AsyncClient

os.environ.setdefault("DATABASE_URL", f"sqlite:///tests.db")


# Primeira função - Retorna o banco de dados
@pytest_asyncio.fixture
async def db(request):
    from src.app.database import database, engine, metadata
    from src.models.post import posts

    await database.connect()
    metadata.create_all(engine)

    def teardown():
        async def _teardown():
            await database.disconnect()
            metadata.drop_all(engine)

        asyncio.run(_teardown())

    request.addfinalizer(teardown)


# Segunda função - Retorna o http cliente
@pytest_asyncio.fixture
async def client(db):
    from src.app.main import app

    transport = ASGITransport(app = app)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    async with AsyncClient(base_url = "http://test", transport= transport, headers= headers) as client:
        yield client



# Terceira função - Facilita o acess_token

## ***1.4. Atualização do arquivo 'conftest.py' na pasta 'tests' com a nova função***

In [ ]:
import asyncio
import os

import pytest_asyncio
from httpx import ASGITransport, AsyncClient

os.environ.setdefault("DATABASE_URL", f"sqlite:///tests.db")


# Primeira função - Retorna o banco de dados
@pytest_asyncio.fixture
async def db(request):
    from src.app.database import database, engine, metadata
    from src.models.post import posts

    await database.connect()
    metadata.create_all(engine)

    def teardown():
        async def _teardown():
            await database.disconnect()
            metadata.drop_all(engine)

        asyncio.run(_teardown())

    request.addfinalizer(teardown)


# Segunda função - Retorna o http client
@pytest_asyncio.fixture
async def client(db):
    from src.app.main import app

    transport = ASGITransport(app = app)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    async with AsyncClient(base_url = "http://test", transport= transport, headers= headers) as client:
        yield client



# Terceira função - Facilita o acess_token
async def acess_token(client: AsyncClient):
    response = await client.post("/auth/login", json={"user_id": 1})
    return response.json()["acess_token"]

----------------------------------------------------------------------------------------------------

# ***2. Definindo o respose para cada vez que o login for efetuado*** 

## ***2.1. Atualização do arquivo 'test_login.py' na pasta 'auth' em 'controllers' da pasta 'integration'***

In [ ]:
from fastapi import status
from httpx import AsyncClient

async def test_login_sucess(client: AsyncClient):
    # Given
    data = {"user_id": 1}

    # When
    response = await client.post("/auth/login", json= data)

    # Then
    assert response.status_code == status.HTTP_200_OK
    assert response.json()["access_token"] is not None

Esses três comentários representam o padrão GWT (Given / When / Then), muito usado em testes. 

É uma forma de organizar o raciocínio do teste de maneira clara e legível:

----------------------------------------------------------------------------------------------------

# ***3. Definindo cada processo de execução teste para o CRUD*** 

Precisamos definir quais os responses para cada situação ao serem executadas.

## ***3.1. Criação do arquivo 'test_create_post.py' na pasta 'post' em 'controllers' da pasta 'integration'***

In [ ]:
from fastapi import status
from httpx import AsyncClient


async def test_create_post_success(client: AsyncClient, access_token: str):
    # Given
    headers = {"Authorization": f"Bearer {access_token}"}
    data = {"title": "post 1", "content": "some content", "published_at": "2026-05-11", "published": True}

    # When
    response = await client.post("/posts/", json= data, headers= headers)

    # Then
    content = response.json()

    assert response.status_code == status.HTTP_201_CREATED
    assert content["id"] is not None


async def test_create_post_invalid_payload_fail(client: AsyncClient, access_token: str):
    # Given
    headers = {"Authorization": f"Bearer {access_token}"}
    data = {"content": "some content", "published_at": "2026-05-11", "published": True}

    # When
    response = await client.post("/posts/", json= data, headers= headers)

    # Then
    content = response.json()


    assert response.status_code == status.HTTP_422_UNPROCESSABLE_CONTENT
    assert content["detail"][0]["loc"] == ["body", "title"]

